# 📗 Sesión 4 · Ejercicio B — Estudiante
## Pestaña de score de riesgo de rotación

**Modalidad:** Independiente — fórmula del score dada, construye la pestaña  
**Tiempo:** 50 minutos  
**Dataset:** IBM HR Analytics (`hr_ibm.csv`)

---

### Objetivo

Agregar al dashboard integrador una **nueva pestaña** llamada "Riesgo de rotación"
que muestre los 10 empleados con mayor probabilidad de abandonar la empresa,
según un score calculado con 3 variables.

### Fórmula del score de riesgo

```python
min_sal = df["MonthlyIncome"].min()
max_sal = df["MonthlyIncome"].max()

df["score_salario"]      = 1 - (df["MonthlyIncome"] - min_sal) / (max_sal - min_sal)
df["score_satisfaccion"] = 1 - (df["JobSatisfaction"] - 1) / 3
df["score_overtime"]     = (df["OverTime"] == "Yes").astype(int)

df["RiesgoScore"] = (df["score_salario"] * 0.4 +
                     df["score_satisfaccion"] * 0.4 +
                     df["score_overtime"] * 0.2)
```

### Lo que debes implementar

| Elemento | Detalle |
|----------|---------|
| Pestaña nueva | `ui.nav_panel("Riesgo de rotación", ...)` en el navbarPage |
| Output 1 | Texto: "Top 10 empleados con mayor riesgo" |
| Output 2 | Tabla con top 10, columnas: Número, Departamento, Puesto, Score, Attrition real |
| Output 3 (opcional) | Gráfica de barras del score por departamento |

> La tabla debe estar ordenada de mayor a menor score.
> El score refleja riesgo — un score de 0.8 significa riesgo muy alto.


## Celda 1 — Instalación y exploración del score

In [1]:
# Instalación y exploración del score de riesgo
import pandas as pd
import plotly.express as px
from shiny import App, ui, render, reactive

# Calcular score de riesgo (fórmula dada — ejecutar para ver el resultado)
df = pd.read_csv("../hr_ibm.csv")

#calcular el riesgo
min_sal = df["MonthlyIncome"].min()
max_sal = df["MonthlyIncome"].max()

df["score_salario"] = 1 - (df["MonthlyIncome"] - min_sal) / (max_sal - min_sal)
df["score_satisfaccion"] = 1 - (df["JobSatisfaction"] - 1)/3
df["score_overtime"] = (df["OverTime"] == "Yes").astype(int)
df["RiesgoScore"] = (df["score_salario"] * 0.4 + df["score_satisfaccion"]*0.4 + df["score_overtime"] * 0.2).round(3)

print("los 10 empleados con mayor riesgo de ayos")
top10 = df.nlargest(10, "RiesgoScore")[
    ["EmployeeNumber", "Department", "JobRole", "RiesgoScore", "Attrition", "MonthlyIncome", "OverTime"]
]

print(top10.to_string(index=False))

los 10 empleados con mayor riesgo de ayos
 EmployeeNumber             Department               JobRole  RiesgoScore Attrition  MonthlyIncome OverTime
            811 Research & Development Laboratory Technician        0.988       Yes           1601      Yes
            133        Human Resources       Human Resources        0.978       Yes           2073      Yes
           1053 Research & Development    Research Scientist        0.978       Yes           2042      Yes
           1569 Research & Development Laboratory Technician        0.978       Yes           2074      Yes
            454 Research & Development Laboratory Technician        0.977       Yes           2119      Yes
           1649 Research & Development Laboratory Technician        0.976       Yes           2166      Yes
           1244 Research & Development    Research Scientist        0.974        No           2235      Yes
           1905 Research & Development    Research Scientist        0.973       Yes           

## Estructura sugerida

La nueva pestaña va dentro del `navbarPage` existente. El score se calcula
dentro de `datos_filtrados()` para que los filtros activos lo afecten.

```python
# En ui — agregar esta pestaña dentro de ui.page_navbar(...)
ui.nav_panel("Riesgo de rotacion",
    ui.output_text("titulo_riesgo"),
    ui.output_ui("tabla_riesgo"),    # usar output_ui para poder colorear filas
    ui.output_ui("grafica_riesgo"),  # opcional: barras de score por dpto
)

# En server — agregar dentro del @reactive.calc
# Calcular RiesgoScore con la fórmula dada sobre el subconjunto filtrado

# Outputs nuevos:
@render.text
def titulo_riesgo():
    return f"Top 10 empleados con mayor riesgo en el grupo filtrado"

@render.ui
def tabla_riesgo():
    # Obtener top 10
    # Construir tabla HTML con colores condicionales por score
    ...
```

> Pista para colorear filas: `style=f"background:{color};"` dentro de `<tr>` en HTML.


## Celda 3 — Tu dashboard extendido

In [ ]:
# Tu extensión del dashboard con la pestaña de riesgo
# Ejecuta la Celda 1 primero para asegurarte de que df_base y min_sal/max_sal existan
import pandas as pd
import plotly.express as px
from shiny import App, ui, render, reactive

# Calcular score de riesgo (fórmula dada — ejecutar para ver el resultado)
df = pd.read_csv("../hr_ibm.csv")

#calcular el riesgo
min_sal = df["MonthlyIncome"].min()
max_sal = df["MonthlyIncome"].max()

opts_dept = ["Todos"] + sorted(df["Department"].unique().tolist())
opts_attr = ["Todos", "Yes", "No"]

css = ui.tags.style("""
    body{ background:#F4F6F9; font-family: Segoe UI, Arial, sans-serif; }
    .navbar{ background-color:#3C3E7A; }
    .navbar-brand, .nav-link{ color:white; font-weight:500; }   
    .kpi-card{ border-radius:10px; padding:20px; text-aling:center; box-shadow:0 2px 8px rgba(0,0,0, 0.1);  }
    .kpi-valor{ font-size:2em, font-weight:bold; }
    .kpi-label{ font-size:0.85em, color:#666; margin-top:4px;}
    h3{ color:#1F3464; border-bottom:2px solid #2E5FA3; padding-bottom:6px; }
    table { width:100%; border-collapse:collapse; font-size:13px; }
    th{ background:#1F3864; color:white; padding: 8px; text-aling: left; }
    td{ padding:7px; border-bottom:0.5px solid #ddd } 
""")

app_ui = ui.page_navbar(
    ui.nav_panel("Si nos dejan ", ui.layout_sidebar(
        ui.sidebar(
            ui.h5("Filtros"),
            ui.input_select("dept", "Departamento", choices=opts_dept),
            ui.input_select("attrition", "Rotación", choices=opts_attr),
            ui.input_slider("edad", "Rango de edad", min=18, max=60, value=[18,60]),
            ui.input_action_button("btn_reset", "Restablecer", class_="btn-sm mt-2")
        ),
        ui.row(
            ui.column(4, ui.output_ui("kpi_total")),
            ui.column(4, ui.output_ui("kpi_rotacion")),
            ui.column(4, ui.output_ui("kpi_salario"))
        ),
        ui.row(
            ui.column(6, ui.output_ui("grafica_rotacion")),
            ui.column(6, ui.output_ui("grafica_scatter"))
        ),
    )),
    ui.nav_panel("Riesgo de rotación y que nos dejen por algo mejor que la pizza", 
                 ui.output_text("titulo_riesgo"),
                 ui.output_ui("tabla_riesgo"),
                 ui.output_ui("grafica_riesgo")
                 ),
    ui.nav_panel("Detalle", ui.output_table("tabla_detalle")),
    title="Dashboard",
    header=css,
)


def server(input, output, session):
    @reactive.calc
    def datos_filtados():
        d = df.copy()
        if input.dept() != "Todos": d = d[d["Department"] == input.dept()]
        if input.attrition() != "Todos": d = d[d["Attrition"] == input.attrition()]
        emin, emax = input.edad()
        d = d[(d["Age"] >= emin ) & (d["Age"] <= emax )]
        d["score_salario"] = 1 - (d["MonthlyIncome"] - min_sal) / (max_sal - min_sal)
        d["score_satisfaccion"] = 1 - (d["JobSatisfaction"] - 1)/3
        d["score_overtime"] = (d["OverTime"] == "Yes").astype(int)
        d["RiesgoScore"] = (d["score_salario"] * 0.4 + d["score_satisfaccion"]*0.4 + d["score_overtime"] * 0.2).round(3)
        return d
    
    @reactive.effect
    @reactive.event(input.btn_reset)
    def _reset():
        ui.update_select("dept", selected="Todos")
        ui.update_select("attrition", selected="Todos")
        ui.update_slider("edad", value=[18,60])

    @render.ui
    def kpi_total():
        n = len(datos_filtados())
        return ui.div(ui.div(f"{n:,}", class_="kpi-valor", style="color:#1F3864"), ui.div("Total empleados: ", class_="kpi-label"), class_="kpi-card")
    
    @render.ui
    def kpi_rotacion():
        d = datos_filtados()
        pct = (d["Attrition"] == "Yes").mean()*100
        col = "#C62862" if pct > 20 else ("#F57F17" if pct > 10 else "#2E7D32")
        return ui.div(ui.div(f"{pct:.1f}%", class_="kpi-valor", style=f"color:{col}"), ui.div("Tasa de rotación: ", class_="kpi-label"), class_="kpi-card")
    
    @render.ui
    def kpi_salario():
        sal = datos_filtados()["MonthlyIncome"].mean()
        return ui.div(ui.div(f"{sal:,.0f}", class_="kpi-valor", style="color:#2E7D32"), ui.div("Salario Promedio: ", class_="kpi-label"), class_="kpi-card")
    
    @render.ui
    def grafica_rotacion():
        d = datos_filtados()
        rot = d.groupby("Department")["Attrition"].apply(lambda x:(x == "Yes").mean()*100).reset_index(name="Rotacion_pct").round(1)
        fig = px.bar(rot, x="Department", y="Rotacion_pct", color="Department", text_auto=".1f", title="Rotación por Departamento (%)")
        fig.update_layout(showlegend=False, height=320)
        return ui.HTML(fig.to_html(include_plotlyjs="cdn"))
    
    @render.ui
    def grafica_scatter():
        d = datos_filtados()
        fig = px.scatter(d, x="MonthlyIncome", y="JobSatisfaction", color="Attrition", color_discrete_map={"Yes":"#E53935", "No":"#43A034"}, hover_data=["JobRole", "Age"], opacity=0.7, title="Salario vs Satifacción")
        fig.update_layout(height=320)
        return ui.HTML(fig.to_html(include_plotlyjs="cdn"))
    
    @render.ui
    def tabla_riesgo():
        d = datos_filtados()
        top10 = d.nlargest(10, "RiesgoScore")[
    ["EmployeeNumber", "Department", "JobRole", "RiesgoScore", "Attrition", "MonthlyIncome", "OverTime"]].reset_index(drop=True)
        rows = ""
        for _, row in top10.iterrows():
            score = row["RiesgoScore"]
            bg = "#FFBEEE" if score > 0.7 else ("#FFF3E0" if score > 0.5 else "#E8F5E9")
            rows += (f"<tr style='background:{bg};'>"
                     f"<td>{int(row['EmployeeNumber'])}</td>"
                     f"<td>{row['Department']}</td>"
                     f"<td>{row['JobRole']}</td>"
                     f"<td>{score:.3f}</td>"
                     f"<td>{row['Attrition']}</td>"
                     f"<td>{int(row['MonthlyIncome']):,}</td>"
                     f"<td>{row['OverTime']}</td>"
                     f"</tr>")
        
        html = ("<table>"
                "<thead><tr>"
                "<th>Emp #</th><th>Departamento</th><th>Puesto</th>"
                "<th>Score riesgo</th><th>Rotación</th><th>Salario</th><th>Horas extra</th>"
                "</tr></thead>"
                f"<tbody>{rows}</tbody>"
                "</table>")
        
        return ui.HTML(html)
    
    @render.ui
    def grafica_riesgo():
        d = datos_filtados()
        agg = d.groupby("Department")["RiesgoScore"].mean().reset_index()
        agg.columns = ["Department", "Score_Promedio"]
        agg = agg.sort_values("Score_Promedio", ascending=False)
        fig = px.bar(agg, x="Department", y="Score_Promedio", color="Score_Promedio", color_continuous_scale="RdYlGn_r", text_auto=".3f", title="Escore de Riesgo por Promedio por Departamento")
        fig.update_layout(showlegend=False, height=300, coloraxis_showscale=False)
        return ui.HTML(fig.to_html(include_plotlyjs="cdn"))
    
    @render.table
    def tabla_detalle():
        cols = ["EmployeeNumber", "Department", "JobRole", "Age", "Attrition", "MonthlyIncome", "RiesgoScore"]
        return datos_filtados().sort_values("RiesgoScore", ascending=False)[cols].head(20)
    
app = App(app_ui, server)

import threading

def _iniciar():
    app.run(host="127.0.0.1", port=8000)

hilo = threading.Thread(target=_iniciar, daemon=True)
hilo.start()
print("App corriendo en local")


App corriendo en local


INFO:     Started server process [14496]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:42513 - "GET / HTTP/1.1" 200 OK


INFO:     127.0.0.1:3028 - "WebSocket /websocket/" [accepted]
INFO:     connection open


## Celda 4 — Preguntas de análisis

In [ ]:
# Preguntas de análisis — usa tu dashboard para explorar

print("Preguntas de análisis — Ejercicio 4B")
print("=" * 45)
print()
print("P1: Abre la pestaña 'Riesgo de rotacion' sin ningún filtro aplicado.")
print("    Observa el top 10. ¿Los empleados pertenecen todos al mismo")
print("    departamento o están distribuidos?")
print()
print("P2: En la columna 'Attrition real' de la tabla,")
print("    ¿cuántos de los 10 empleados con mayor score SÍ rotaron realmente?")
print("    ¿Crees que el score predice bien?")
print()
print("P3: Observa los valores de los 3 componentes del score")
print("    (salario, satisfacción, overtime) en las filas rojas.")
print("    ¿Cuál de las 3 variables parece tener más peso en los casos críticos?")
print()
print("Escribe tus respuestas (convierte a Markdown):")
print("P1: ...")
print("P2: ...")
print("P3: ...")


Preguntas de análisis — Ejercicio 4B

P1: Abre la pestaña 'Riesgo de rotacion' sin ningún filtro aplicado.
    Observa el top 10. ¿Los empleados pertenecen todos al mismo
    departamento o están distribuidos?

P2: En la columna 'Attrition real' de la tabla,
    ¿cuántos de los 10 empleados con mayor score SÍ rotaron realmente?
    ¿Crees que el score predice bien?

P3: Observa los valores de los 3 componentes del score
    (salario, satisfacción, overtime) en las filas rojas.
    ¿Cuál de las 3 variables parece tener más peso en los casos críticos?

Escribe tus respuestas (convierte a Markdown):
P1: ...
P2: ...
P3: ...


INFO:     Started server process [18952]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:46345 - "GET / HTTP/1.1" 200 OK


INFO:     127.0.0.1:48328 - "WebSocket /websocket/" [accepted]
INFO:     connection open
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\Lib\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'score'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\demon\AppData\Roaming\Python\Python312\site-packages\shiny\session\_session.py", line 2261, in output_obs
    value = await renderer.render()
            ^^^^^^^^^^^^^^